# SeismicDataset. Часть 1: подготовка и исследование моделей #

## Глава 1.4. Подготовка кафедрального сервера к расчёту датасета ##

Цель главы — подготовить воспроизводимое выполнение датасета на кафедральном сервере `antonov@188.44.53.153`, доступном с текущей рабочей станции. Первый этап ограничен проверкой SSH, характеристик, загрузки и хранения. На сервере не устанавливались программы, не изменялись конфигурация, данные, службы и задания; решатель и тесты производительности не запускались. Доступ с других машин не проверялся: ограничение «только с этой машины» принято как условие пользователя.

Этот дневник читает только локальные сохранённые свидетельства. Его выполнение не устанавливает SSH-соединение и не запускает удалённые команды. Снимок отражает 17 ноября 2026 года, 20:08–20:10 МСК, а не постоянную доступность общих ресурсов.

In [1]:
import hashlib
import json
from pathlib import Path
import xml.etree.ElementTree as ET

from scripts.project_paths import PROJECT_ROOT
from scripts.reporting import mapping, number, read_json, sequence

In [2]:
audit_root = PROJECT_ROOT / "data/dev_1_4_server_preparation/server_audit_20261117_v1"
snapshot = read_json(audit_root / "snapshot.json")
manifest = read_json(audit_root / "manifest.json")
supplement = read_json(audit_root / "supplement.json")
commands = mapping(snapshot["commands"])
extra_commands = mapping(supplement["commands"])
assert hashlib.sha256((audit_root / "snapshot.json").read_bytes()).hexdigest() == manifest["snapshot_sha256"]
assert hashlib.sha256((audit_root / "probe.py.txt").read_bytes()).hexdigest() == manifest["probe_sha256"]
print("Снимок UTC:", snapshot["started_utc"], "—", snapshot["finished_utc"])
print("SSH:", mapping(commands["hostname"])["stdout"], mapping(commands["identity"])["stdout"])

Снимок UTC: 2026-11-17T17:08:55.716774+00:00 — 2026-11-17T17:09:15.144385+00:00
SSH: saturn
 uid=1013(antonov) gid=1002(antonov) groups=1002(antonov),27(sudo),998(docker),1000(pikman)



## 1. Паспорт оборудования ##

Сведения получены из lscpu, /proc/meminfo, DMI через sysfs, lsblk и nvidia-smi; привилегированные средства не использовались. Сервер HPE ProLiant DL380 Gen10 имеет два Xeon Silver 4215R, всего 16 физических ядер и 32 потока, два NUMA-узла. ОС видит около 503,56 ГиБ RAM; точная конфигурация DIMM отдельно не установлена.

In [3]:
cpu = mapping(json.loads(str(mapping(commands["cpu"])["stdout"])))
cpu_fields = {str(mapping(item)["field"]): mapping(item)["data"] for item in sequence(cpu["lscpu"])}
for field in ("Model name:", "Socket(s):", "Core(s) per socket:", "Thread(s) per core:", "CPU(s):", "CPU max MHz:", "NUMA node(s):"):
    print(field, cpu_fields[field])
memory_lines = str(mapping(commands["memory"])["stdout"]).splitlines()
memory_kib = {line.split(":")[0]: int(line.split()[1]) for line in memory_lines}
for field in ("MemTotal", "MemAvailable", "SwapTotal", "SwapFree"):
    print(f"{field}: {memory_kib[field] / 2**20:.3f} ГиБ")
gpu_xml = ET.fromstring(str(mapping(commands["gpu"])["stdout"]))
for index, gpu in enumerate(gpu_xml.findall("gpu")):
    print(f"GPU {index}: {gpu.findtext('product_name')}, память {gpu.findtext('fb_memory_usage/total')}, свободно {gpu.findtext('fb_memory_usage/free')}")
    print("  ECC:", gpu.findtext("ecc_mode/current_ecc"), "; MIG:", gpu.findtext("mig_mode/current_mig"))
    print("  Накопленные DRAM ECC corrected/uncorrected:", gpu.findtext("ecc_errors/aggregate/dram_correctable"), gpu.findtext("ecc_errors/aggregate/dram_uncorrectable"))
print(mapping(extra_commands["compute_capability"])["stdout"])
print(mapping(commands["gpu_topology"])["stdout"])

Model name: Intel(R) Xeon(R) Silver 4215R CPU @ 3.20GHz
Socket(s): 2
Core(s) per socket: 8
Thread(s) per core: 2
CPU(s): 32
CPU max MHz: 4000.0000
NUMA node(s): 2
MemTotal: 503.556 ГиБ
MemAvailable: 485.577 ГиБ
SwapTotal: 8.000 ГиБ
SwapFree: 8.000 ГиБ
GPU 0: NVIDIA A100-PCIE-40GB, память 40960 MiB, свободно 40332 MiB
  ECC: Enabled ; MIG: Disabled
  Накопленные DRAM ECC corrected/uncorrected: 830 0
GPU 1: NVIDIA A100-PCIE-40GB, память 40960 MiB, свободно 40332 MiB
  ECC: Enabled ; MIG: Disabled
  Накопленные DRAM ECC corrected/uncorrected: 0 0
index, compute_cap
0, 8.0
1, 8.0

	GPU0	GPU1	CPU Affinity	NUMA Affinity
GPU0	 X 	SYS	0-7,16-23	0
GPU1	SYS	 X 	8-15,24-31	1

Legend:

  X    = Self
  SYS  = Connection traversing PCIe as well as the SMP interconnect between NUMA nodes (e.g., QPI/UPI)
  NODE = Connection traversing PCIe as well as the interconnect between PCIe Host Bridges within a NUMA node
  PHB  = Connection traversing PCIe as well as a PCIe Host Bridge (typically the CPU)
  PXB

Обе карты — A100-PCIE-40GB, compute capability 8.0, лимит мощности каждой 250 Вт. Фактическое соединение PCIe Gen3 ×16; карта поддерживает Gen4, но хост сообщает максимум Gen3. GPU0 относится к NUMA0, GPU1 к NUMA1, связь между ними — SYS, не NVLink. Это две отдельные памяти по 40 ГиБ, а не единый пул на 80 ГиБ. Для независимых моделей планируется отдельный процесс на каждой карте; эффективность ещё не измерена.

ECC включён, MIG выключен. На GPU0 накоплено 830 исправленных DRAM ECC-событий; неисправленных — 0, текущие volatile-счётчики — 0. На GPU1 все указанные счётчики нулевые. По одному снимку нельзя установить время возникновения или динамику этих событий; сохранён исходный счётчик для последующей сверки, а не диагноз неисправности.

## 2. Хранение и доля пользователя ##

Доступный обычному пользователю объём берётся из Available в df, а не из разности общей ёмкости и Used: ext4 оставляет резерв. /home находится на системном разделе. Другие обнаруженные диски не смонтированы, их содержимое и назначение не проверялись; включать их ёмкость в доступное хранилище проекта нельзя.

In [4]:
print(mapping(commands["filesystems"])["stdout"])
disks = mapping(json.loads(str(mapping(commands["disks"])["stdout"])))
for value in sequence(disks["blockdevices"]):
    disk = mapping(value)
    if disk["type"] == "disk":
        print(disk["name"], disk["model"], f"{number(disk['size']) / 10**12:.6f} ТБ", "вращающийся:" , disk["rota"], "интерфейс:", disk["tran"])
usage = mapping(snapshot["home_usage"])
assert usage["complete"] is True
crosscheck = mapping(extra_commands["home_crosscheck"])
assert crosscheck["returncode"] == 0
assert int(str(crosscheck["stdout"]).split()[0]) == int(number(usage["allocated_bytes"]))
print(f"Домашний каталог: {number(usage['allocated_bytes']) / 2**30:.6f} ГиБ")
print(f"Из них UID 1013: {number(usage['antonov_allocated_bytes']) / 2**30:.6f} ГиБ")
print("Уникальных inode:", usage["unique_inodes"], "; секунд обхода:", usage["elapsed_seconds"])
directories = mapping(usage["top_level"])
largest = sorted(directories.items(), key=lambda item: number(mapping(item[1])["allocated_bytes"]), reverse=True)[:10]
for name, record in largest:
    print(f"{name}: {number(mapping(record)['allocated_bytes']) / 2**30:.3f} ГиБ")

Filesystem     Type     1B-blocks          Used    Available Use% Mounted on
/dev/sda2      ext4 1887976378368 1614166011904 177830903808  91% /
/dev/sda1      vfat    1124999168       6377472   1118621696   1% /boot/efi

sda VK001920GXCGP 1.920383 ТБ вращающийся: False интерфейс: sas
sdb VK001920GXCGP 1.920383 ТБ вращающийся: False интерфейс: sas
sdc VK001920GXCGP 1.920383 ТБ вращающийся: False интерфейс: sas
sdd VK001920GXCGP 1.920383 ТБ вращающийся: False интерфейс: sas
sde EG002400JWJNT 2.400477 ТБ вращающийся: True интерфейс: sas
sdf EG002400JWJNT 2.400477 ТБ вращающийся: True интерфейс: sas
Домашний каталог: 115.836685 ГиБ
Из них UID 1013: 102.943348 ГиБ
Уникальных inode: 250098 ; секунд обхода: 11.632611197885126
TrySEMGPU: 40.163 ГиБ
ProgramSeismic3D: 20.104 ГиБ
abc_dataset: 12.893 ГиБ
beta: 7.991 ГиБ
gamma: 7.682 ГиБ
fidesys: 6.000 ГиБ
.vscode-server: 4.875 ГиБ
alpha: 3.025 ГиБ
semgpu-test: 2.619 ГиБ
abc_dataset.tar.gz: 2.578 ГиБ


На /dev/sda2: 1887976378368 байт общей ёмкости, 1614166011904 байт занято, 177830903808 байт доступно пользователю — 165,618 ГиБ (177,831 ГБ), заполнение по df 91%. Физически видны четыре SAS SSD VK001920GXCGP по 1,920 ТБ и два SAS HDD EG002400JWJNT по 2,400 ТБ. Только sda содержит смонтированные разделы; sdb–sdf не обследовались на наличие данных и не подключались.

В /home/antonov учтено 124378693632 байт (115,837 ГиБ) выделенных блоков, включая каталоги и ссылки; из них UID 1013 принадлежит 110534578176 байт (102,943 ГиБ). Разность 13844115456 байт (12,893 ГиБ) находится в abc_dataset с другим владельцем; корневой каталог принадлежит pikman. Обычный du независимо дал точно тот же общий объём. Обход 250098 уникальных inode завершился без ошибок за 11,63 с, при nice=19 и idle I/O priority, без перехода по символьным ссылкам и на другие файловые системы.

Это полный учёт домашнего каталога, но нижняя граница глобального объёма файлов UID 1013: quota отсутствует, чужие домашние каталоги и Docker-хранилище не сканировались. В /var/tmp найдены три собственных swap-файла редактора, их объём в эти числа не включён. Системное использование и контейнерные данные, принадлежащие root, не приписывались пользователю. Ничего не удалялось и владельцы файлов не менялись.

## 3. Загрузка во время проверки ##

Нагрузка оценивается короткими последовательными наблюдениями; первая строка vmstat усреднена с загрузки ОС, следующие три относятся к секундным интервалам. Состояние общего сервера может измениться сразу после проверки, а отсутствие загрузки не означает резервирование ресурсов.

In [5]:
print(mapping(commands["uptime"])["stdout"])
print(mapping(commands["vmstat"])["stdout"])
for value in sequence(snapshot["gpu_samples"]):
    print(mapping(value)["stdout"])
print("После аудита:", mapping(extra_commands["load_after"])["stdout"])

 17:08:55 up 48 days, 10:31,  1 user,  load average: 0.14, 0.12, 0.05

procs -----------memory---------- ---swap-- -----io---- -system-- ------cpu-----
 r  b   swpd   free   buff  cache   si   so    bi    bo   in   cs us sy id wa st
 0  0      0 489546464 2835684 20436916    0    0     0    11    0    0  1  0 99  0  0
 0  0      0 489543904 2835684 20438536    0    0     0    24  537 1907  0  0 100  0  0
 0  0      0 489543616 2835684 20438560    0    0     0   552  901 2845  0  0 100  0  0
 0  0      0 489543712 2835684 20438560    0    0     0  1160  917 3350  1  0 99  0  0

timestamp, index, name, memory.total [MiB], memory.used [MiB], utilization.gpu [%], utilization.memory [%], temperature.gpu, power.draw [W], power.limit [W]
2026/11/17 17:08:59.337, 0, NVIDIA A100-PCIE-40GB, 40960 MiB, 5 MiB, 0 %, 0 %, 43, 38.71 W, 250.00 W
2026/11/17 17:08:59.344, 1, NVIDIA A100-PCIE-40GB, 40960 MiB, 5 MiB, 0 %, 0 %, 34, 34.76 W, 250.00 W

timestamp, index, name, memory.total [MiB], memory.used 

В начале load average 0,14 / 0,12 / 0,05 на 32 логических CPU; в конце 0,40 / 0,20 / 0,08, включая вклад самого обследования. В трёх секундных интервалах CPU простаивал на 99–100%, I/O wait 0%, swap не использовался. Доступно 485,577 ГиБ RAM. На обеих GPU в трёх снимках за четыре секунды загрузка 0%, занято по 5 МиБ, вычислительных процессов в XML нет; графическая служба Xorg присутствует. Температуры 43 °C и 34–35 °C, потребление примерно 39 и 35 Вт.

У antonov существуют старые процессы python3/start, не потреблявшие заметного CPU в снимке; их назначение не исследовалось и они не останавливались. Команды Slurm/PBS не обнаружены в PATH, но это не доказывает отсутствие кафедральных правил бронирования или другого планировщика.

## 4. Первоначальная оценка хранения четырёх компонент — заменена решением раздела 5 ##

По дополнительному поручению оценивается хранение всех 60000 моделей при p4, пяти источниках и N приёмниках на каждый источник. Это оценка объёма, не запуск полной серии. Сохраняются только Ux, Uy, Vx, Vy, по 2001 отсчёту на [0; 2] с, как в главе 1.3. Принят действующий float64: 8 байт на скаляр. Сжатие в гарантированную оценку не заложено; N — число приёмников, а не внутренних узлов сетки.

На один приёмник одного источника одной модели:

$$
B_{\mathrm{receiver}}=2001\cdot4\cdot8=64032\ \text{байта}=62{,}53125\ \text{КиБ}.
$$

Это 32 байта на один временной отсчёт четырёх компонент. При записи пяти источников одной модели требуется 320160 байт на приёмник. На весь датасет вклад одного приёмника составляет 19,2096 ГБ.

$$
V_{\mathrm{signals}}(N)=60000\cdot5\cdot N\cdot2001\cdot4\cdot8=19209600000N\ \text{байт}=0{,}0192096N\ \text{ТБ}.
$$

Для одного геометрического варианта каждой модели и прежней линии N=141 (x=0,5,…,700 м):

$$
V_{\mathrm{signals}}(141)=2708553600000\ \text{байт}=2{,}7085536\ \text{ТБ}\approx2{,}4634\ \text{ТиБ}.
$$

Порядок p4 влияет на внутренние массивы, шаг времени и время расчёта, но не входит в объём фиксированной записи приёмников. Два геометрических варианта каждой модели, например raster и curved, удвоят указанное хранение. Это также не оценка необходимой RAM/VRAM решателя: 62,53 КиБ — только буфер сейсмограммы одного приёмника для одного опыта.

In [6]:
models_count = 60000
sources_count = 5
time_samples = 2001
receiver_components = 4
scalar_bytes = 8
receiver_count = 141
bytes_per_receiver = time_samples * receiver_components * scalar_bytes
signal_coefficient = models_count * sources_count * bytes_per_receiver
signal_bytes = signal_coefficient * receiver_count
# Existing surface.npz also stores time, XY coordinates and int64 node IDs per run.
axes_and_ids_bytes = models_count * sources_count * (time_samples * 8 + receiver_count * 24)
surface_array_bytes = signal_bytes + axes_and_ids_bytes
print("Приёмник × источник × модель:", bytes_per_receiver, "байт;", bytes_per_receiver / 1024, "КиБ")
print("Коэффициент V(N):", signal_coefficient, "байт на приёмник по всему датасету")
print("N=141: сигналы", signal_bytes, "байт =", signal_bytes / 10**12, "ТБ")
print("Оси и ID при повторении в каждом опыте:", axes_and_ids_bytes / 10**9, "ГБ")
print("Все массивы surface без сжатия и заголовков:", surface_array_bytes / 10**12, "ТБ")
print("Иллюстративный бюджет +20%:", 1.2 * surface_array_bytes / 10**12, "ТБ")

Приёмник × источник × модель: 64032 байт; 62.53125 КиБ
Коэффициент V(N): 19209600000 байт на приёмник по всему датасету
N=141: сигналы 2708553600000 байт = 2.7085536 ТБ
Оси и ID при повторении в каждом опыте: 5.8176 ГБ
Все массивы surface без сжатия и заголовков: 2.7143712 ТБ
Иллюстративный бюджет +20%: 3.25724544 ТБ


В существующем surface.npz дополнительно находятся временная ось float64 (2001 чисел), координаты XY float64 и node_ids int64 (по N). Если повторять их для каждого из 300000 опытов, объём всех массивов будет:

$$
V_{\mathrm{surface}}(N)=300000\,[64032N+16008+24N]=19216800000N+4802400000\ \text{байт}.
$$

При N=141 это 2,7143712 ТБ, из которых оси и ID занимают 5,8176 ГБ. Заголовки NPZ/NPY, метаданные опытов и накладные расходы файловой системы добавляются сверх этой величины; сжатие может её уменьшить, но коэффициент заранее неизвестен. Общие оси можно хранить один раз, а идентификаторы узлов и координаты — один раз на геометрию. Практический ориентир с условным запасом 20% — около 3,26 ТБ; этот запас является плановым допущением, а не измеренной потребностью. Входы, сетки, среда и резервные копии требуют отдельного места и в этот бюджет не включены.

Доступные сейчас 177,831 ГБ системного раздела меньше массива сигналов при N=141 примерно в 15,2 раза. Следовательно, разместить весь результат там нельзя даже без полевых кадров. Флаг --no-fields текущего исследовательского запускателя всё ещё сохраняет source_history на каждом внутреннем шаге; массовый режим «только сейсмограммы» предстоит выделить отдельно, чтобы такая диагностика не создавала незапланированный объём. Оценка не предполагает сохранения всех сеток, источниковых историй и анимаций для всех 300000 опытов.

## 5. Принятое сокращение записи и расчёт чанками ##

После первоначальной оценки пользователь сократил запись до 70 приёмников в центрах прежних 10-метровых ячеек, только Vx и Vy. Координаты x=5,15,…,695 м, y=0; крайние x=0 и x=700 в новую запись не входят. Эти координаты являются вершинами уточнённой верхней сетки с шагом 5 м. Сохраняются 2001 отсчёт за 2 с и float64; точность хранения не изменяется. p4 и пять источников остаются параметрами планируемой полной серии.

Новый объём одного приёмника одного источника:

$$
B^{(V)}_{\mathrm{receiver}}=2001\cdot2\cdot8=32016\ \text{байт}=31{,}265625\ \text{КиБ}.
$$

$$
V^{(V)}(N)=60000\cdot5\cdot N\cdot2001\cdot2\cdot8=9604800000N\ \text{байт}.
$$

При N=70 требуется 2241120 байт на источник, 11205600 байт на полную модель из пяти источников, а на весь датасет — 672336000000 байт = 672,336 ГБ. Это в 4,029 раза меньше первоначальной записи 141×4. Сжатие не учитывается. Чанки уменьшают одновременное хранение на сервере, но не суммарный объём конечного датасета.

In [7]:
accepted_receivers_x_m = list(range(5, 700, 10))
accepted_components = ("Vx", "Vy")
assert len(accepted_receivers_x_m) == 70
velocity_bytes_per_receiver = time_samples * len(accepted_components) * scalar_bytes
velocity_bytes_per_source = len(accepted_receivers_x_m) * velocity_bytes_per_receiver
velocity_bytes_per_model = sources_count * velocity_bytes_per_source
velocity_bytes_total = models_count * velocity_bytes_per_model
# Conservative repetition of the same auxiliary arrays as the current surface format.
velocity_aux_bytes_per_model = sources_count * (time_samples * 8 + len(accepted_receivers_x_m) * 24)
velocity_surface_bytes_per_model = velocity_bytes_per_model + velocity_aux_bytes_per_model
print("Принятые приёмники:", accepted_receivers_x_m)
print("Компоненты:", accepted_components, "; байт на приёмник одного опыта:", velocity_bytes_per_receiver)
print("На источник:", velocity_bytes_per_source, "байт; на модель с 5 источниками:", velocity_bytes_per_model, "байт")
print("Все 60000 моделей, только сигналы:", velocity_bytes_total / 10**9, "ГБ")
for chunk_models in (1000, 5000, 10000):
    print(f"Чанк {chunk_models} моделей: сигналы {chunk_models * velocity_bytes_per_model / 10**9:.6f} ГБ; с повторяемыми осями/ID {chunk_models * velocity_surface_bytes_per_model / 10**9:.6f} ГБ")

Принятые приёмники: [5, 15, 25, 35, 45, 55, 65, 75, 85, 95, 105, 115, 125, 135, 145, 155, 165, 175, 185, 195, 205, 215, 225, 235, 245, 255, 265, 275, 285, 295, 305, 315, 325, 335, 345, 355, 365, 375, 385, 395, 405, 415, 425, 435, 445, 455, 465, 475, 485, 495, 505, 515, 525, 535, 545, 555, 565, 575, 585, 595, 605, 615, 625, 635, 645, 655, 665, 675, 685, 695]
Компоненты: ('Vx', 'Vy') ; байт на приёмник одного опыта: 32016
На источник: 2241120 байт; на модель с 5 источниками: 11205600 байт
Все 60000 моделей, только сигналы: 672.336 ГБ
Чанк 1000 моделей: сигналы 11.205600 ГБ; с повторяемыми осями/ID 11.294040 ГБ
Чанк 5000 моделей: сигналы 56.028000 ГБ; с повторяемыми осями/ID 56.470200 ГБ
Чанк 10000 моделей: сигналы 112.056000 ГБ; с повторяемыми осями/ID 112.940400 ГБ


Предлагаемая единица чанка — полная модель со всеми пятью источниками, без разделения её записей между чанками. Для C моделей в одном чанке объём сигналов равен 11205600C байт. Если повторять временную ось, XY и int64 node_ids для каждого источника, все массивы занимают 11294040C байт; это консервативный вариант, а не окончательно утверждённый формат упаковки. Например, 1000 моделей дают 11,206 ГБ сигналов или 11,294 ГБ вместе с такими служебными массивами; 10000 моделей — 112,056 / 112,940 ГБ соответственно.

После очистки пользователь сообщит о готовности; затем потребуется новый read-only замер Available. Размер чанка пока не назначен. Пусть F — фактически доступные байты на выделенном файловом разделе, R — согласованный неприкосновенный резерв, W — место для среды, входов, рабочих сеток, временных файлов и упаковки. Тогда предварительная верхняя граница числа полных моделей в чанке при текущем варианте служебных массивов:

$$
C_{\max}=\max\left(0,\left\lfloor\frac{F-R-W}{11294040}\right\rfloor\right).
$$

Заголовки, журнал и файловые накладные расходы включаются в W либо уменьшают выбранный C. Формула предполагает один хранимый чанк; если расчёт следующего перекрывается с выгрузкой предыдущего, нужно место для обоих. Сжатие и архивирование не должны незаметно создавать вторую полную копию. Точный C выбирается после замера и проверки рабочего места на небольшом опыте, а не из прежних 165,6 ГиБ Available.

Для освобождения места между чанками потребуется отдельное конечное хранилище: полностью записать чанк, проверить число моделей/источников, формы массивов и контрольные суммы, выгрузить и подтвердить сохранность копии. Удаление исходного чанка с сервера требует согласованного правила; сейчас ничего не удаляется. Ни направление выгрузки, ни автоматическое удаление этим обсуждением не установлены. Запускатель «только Vx/Vy, 70 приёмников» и возобновляемая очередь чанков пока не реализованы; исторические записи 141×4 остаются неизменными.

## 6. Что занимает место: дополнительный аудит перед очисткой ##

По следующему поручению пользователя отдельно оценены собственные каталоги и чужие домашние каталоги. Снимок выполнен 17.11.2026 в 20:20–20:21 МСК, уточнение собственных подкаталогов — позже в том же сеансе. Читались только метаданные размеров и владельцев; содержимое файлов не читалось, sudo не использовался, ничего не удалялось. du работал при nice=19 и idle I/O, каждый запрос ограничен по времени. Закрытые чужие подкаталоги не обходились с повышением прав.

In [8]:
storage_root = PROJECT_ROOT / "data/dev_1_4_server_preparation/storage_audit_20261117_v1"
storage_report = read_json(storage_root / "report.json")
print("Состояние:", storage_report["status"], "; UTC:", storage_report["snapshot_utc"])
print("Собственный домашний каталог, ГиБ:", number(storage_report["own_home_allocated_bytes"]) / 2**30)
print("Видимая часть других домашних каталогов, ГиБ:", number(storage_report["other_home_observed_total_bytes"]) / 2**30)
for item in sequence(storage_report["other_homes"])[:8]:
    entry = mapping(item)
    qualifier = "не менее" if entry["incomplete"] else "видимый учёт"
    print(entry["path"], "владелец", entry["owner"], qualifier, f"{number(entry['observed_allocated_bytes']) / 2**30:.3f} ГиБ")
own_sizes = mapping(storage_report["own_directories"])
for path in ("TrySEMGPU/research/seismic", "TrySEMGPU/research/lave_wave", "ProgramSeismic3D/VTK/build", "ProgramSeismic3D/data/src", ".vscode-server", ".nv/ComputeCache", ".cache"):
    full_path = "/home/antonov/" + path
    print(full_path, f"{number(own_sizes[full_path]) / 2**30:.3f} ГиБ")
print("Собственные архивы >100 МиБ:", f"{number(storage_report['own_large_archive_allocated_bytes']) / 2**30:.3f} ГиБ")

Состояние: completed_with_permission_limits ; UTC: 2026-11-17T17:21:15.525298+00:00
Собственный домашний каталог, ГиБ: 115.83668518066406
Видимая часть других домашних каталогов, ГиБ: 764.7881736755371
/home/vershinin владелец vershinin не менее 187.235 ГиБ
/home/murusidze владелец murusidze не менее 181.975 ГиБ
/home/filatova владелец filatova не менее 176.696 ГиБ
/home/skvortsov владелец skvortsov не менее 127.977 ГиБ
/home/masalkin владелец masalkin не менее 38.144 ГиБ
/home/aristeev владелец aristeev не менее 29.918 ГиБ
/home/yuri владелец yuri не менее 15.908 ГиБ
/home/melnikov владелец liudmila не менее 4.747 ГиБ
/home/antonov/TrySEMGPU/research/seismic 22.396 ГиБ
/home/antonov/TrySEMGPU/research/lave_wave 13.506 ГиБ
/home/antonov/ProgramSeismic3D/VTK/build 7.739 ГиБ
/home/antonov/ProgramSeismic3D/data/src 7.384 ГиБ
/home/antonov/.vscode-server 4.875 ГиБ
/home/antonov/.nv/ComputeCache 0.800 ГиБ
/home/antonov/.cache 0.490 ГиБ
Собственные архивы >100 МиБ: 22.576 ГиБ


В чужих домашних каталогах видно суммарно не менее 764,788 ГиБ. Крупнейшие: vershinin ≥187,235 ГиБ, murusidze ≥181,975 ГиБ, filatova ≥176,696 ГиБ, skvortsov ≥127,977 ГиБ; эти четыре дают ≥673,883 ГиБ. Следом masalkin ≥38,144 ГиБ, aristeev ≥29,918 ГиБ, yuri ≥15,908 ГиБ. У перечисленных пользователей есть закрытые подкаталоги, поэтому это нижние границы, не квоты и не полный учёт UID. По логину нельзя определить, кто является студентом пользователя; содержимое чужих работ не исследовалось.

У antonov по-прежнему 115,837 ГиБ в домашнем каталоге. TrySEMGPU занимает 40,163 ГиБ: research/seismic 22,396 ГиБ, research/lave_wave 13,506 ГиБ, .git 3,372 ГиБ. В seismic два крупных расчётных каталога занимают 8,409 и 8,340 ГиБ, рядом есть архив 3,771 ГиБ; их содержательная эквивалентность и наличие резервных копий не проверены. Историю .git не следует считать обычным кэшем: её файлы не удалялись и перепаковка не выполнялась.

ProgramSeismic3D занимает 20,104 ГиБ: сборка VTK/build 7,739 ГиБ и VTK/build.zip 1,607 ГиБ, вместе 9,346 ГиБ; data/src 7,384 ГиБ, включая profile_model_with_bc_9.dat размером 7,265 ГиБ. Последний является файлом исходных данных по расположению, а не автоматически удаляемым результатом. Данные beta/data/storage и gamma/data/storage занимают 5,109 и 4,660 ГиБ соответственно.

Первые места для ручной проверки возможности очистки: сборка VTK и её архив, старые серверные версии редактора, кэши и архивы уже сохранённых опытов. .vscode-server занимает 4,875 ГиБ; cli/servers содержит семь каталогов выпусков/установки, суммарно 4,354 ГиБ, включая .staging. Какая версия нужна активному подключению, не устанавливалось. .nv/ComputeCache — 0,800 ГиБ, .cache — 0,490 ГиБ, из них pip 0,467 ГиБ.

Найдено 13 собственных архивов размером более 100 МиБ, всего 22,576 ГиБ. В частности, четыре fidesys.tar.gz в корне и alpha/beta/gamma занимают вместе около 7,16 ГиБ; одинаковое назначение имён не доказывает одинаковое содержимое. Архивы входят в размеры перечисленных каталогов, поэтому эти числа нельзя складывать как независимый потенциал освобождения. abc_dataset (12,893 ГиБ) внутри домашнего каталога имеет другого владельца и требует отдельного решения, хотя архив abc_dataset.tar.gz (2,578 ГиБ) принадлежит antonov.

Для контекста видимая часть /usr занимает 149,169 ГиБ, /var 61,749 ГиБ, /opt 10,285 ГиБ; есть запреты доступа, в том числе к контейнерным данным, поэтому полный системный объём не установлен. Разность общего df и собственного каталога нельзя целиком приписать студентам: туда входят ОС, другие пользователи, закрытые данные и служебные расходы файловой системы. В конце обследования Available составлял 177829662720 байт, до пользовательской очистки; утверждённого размера чанка по-прежнему нет.

## 7. Программная среда и следующий этап ##

ОС — Ubuntu 20.04.6 LTS, ядро 5.4.0-216-generic. Фактический драйвер NVIDIA 525.147.05; строка CUDA Version в nvidia-smi сообщает 12.0 и не заменяет версию установленного компилятора. В PATH nvcc 10.1.243, Python 3.8.10, GCC 9.4.0 и CMake 3.16.3. Дополнительная проверка обнаружила CUDA SDK 12.0.1 и 13.0.0 в /usr/local, а также пользовательские каталоги pyenv 3.13.5 и semgpu; их пригодность для текущего проекта пока не проверена.

Локальный профиль главы 1.3 собран с CUDA 12.8 для архитектуры 86, а серверные карты сообщают 8.0. Прямой перенос готовой native-библиотеки не принимается как подтверждённо рабочий вариант. Предстоит выбрать явные пути инструментов, собрать semgpu2 для архитектуры 80 в отдельной пользовательской среде и проверить согласование с локальным эталоном. Системные альтернативы, драйвер и существующие окружения других задач менять не планируется.

In [9]:
print("Драйвер:", gpu_xml.findtext("driver_version"), "; nvidia-smi CUDA:", gpu_xml.findtext("cuda_version"))
for name in ("cuda_compiler", "python", "compiler", "cmake"):
    print(name, str(mapping(commands[name])["stdout"]).splitlines()[0])
print("Пользовательские версии Python:", mapping(extra_commands["pyenv_versions"])["stdout"])
local_profile = read_json(PROJECT_ROOT / "data/dev_1_3_point_source/runtime_combined_v1/profile.json")
print("Локальная опора главы 1.3:", mapping(local_profile["build"])["nvcc"], "архитектура", mapping(local_profile["build"])["cuda_architectures"])

Драйвер: 525.147.05 ; nvidia-smi CUDA: 12.0
cuda_compiler nvcc: NVIDIA (R) Cuda compiler driver
python Python 3.8.10
compiler gcc (Ubuntu 9.4.0-1ubuntu1~20.04.2) 9.4.0
cmake cmake version 3.16.3
Пользовательские версии Python: 3.13.5
semgpu

Локальная опора главы 1.3: /usr/local/cuda-12.8/bin/nvcc архитектура 86


### План дальнейшей главы 1.4 — ещё не выполнен

1. Уточнить выделенный проекту каталог, допустимый объём и порядок совместного использования GPU. Свободные 165,6 ГиБ на системном разделе не являются выделенной квотой датасета; несмонтированные диски требуют отдельного решения ответственного за сервер.
2. Зафиксировать версии исходников, Python-зависимостей и native-профиля; подготовить изолированную среду antonov с явным выбором CUDA и архитектуры 80. Существующий ~/TrySEMGPU оставить отдельной исторической копией до проверки его состояния.
3. По отдельному поручению выполнить короткий тест на одной карте и сравнить с главой 1.3, затем проверить независимый запуск на второй. Для исследования сглаживания рабочая рекомендация остаётся p4; проверка p3 не изменила её.
4. Измерить время и объём результата на небольшой согласованной выборке. Реализовать принятые выходы Vx/Vy на 70 приёмниках и очередь чанков из целых моделей с возобновлением, контролем места и выгрузкой на согласованное хранилище. Размер чанка определить после очистки и нового замера. Полевые анимации главы 1.3 относятся к тестам и не должны автоматически масштабироваться на весь датасет.
5. Полную обработку датасета запускать после отдельного прямого поручения. Для оценки хранения принято новое задание 60000 моделей × 5 источников, только сейсмограммы; это не поручение начать массовый расчёт. Последние выполненные опыты главы 1.3 охватывали только центральный источник.

### Вывод

SSH с рабочей станции подтверждён; обе A100 и CPU во время проверки почти свободны, памяти достаточно для подготовки пилотов. Главные открытые вопросы — выделенное хранилище и воспроизводимая программная среда. Массовый расчёт, резервирование ресурсов и производительность пока не проверены. Сохранённый аудит отделяет доступный пользователю диск от физических несмонтированных устройств и объём домашнего каталога от глобального учёта UID.